# Классификация: SI > 8

Задача: предсказать, превышает ли SI значение порога SI = 8 (практически значимый уровень).

## 1. Импорты

In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedGroupKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

## 2. Загрузка и подготовка

In [2]:
raw_data = pd.read_excel("../data/dataset.xlsx")
TARGET_COLUMNS = ["IC50, mM", "CC50, mM", "SI"]

data = raw_data.iloc[:, 1:].copy()
data = data.drop_duplicates()

feature_columns = [col for col in data.columns if col not in TARGET_COLUMNS]
X_full = data[feature_columns]

constant_mask = X_full.nunique(dropna=False) <= 1

dominant_share = X_full.apply(
    lambda column: column.value_counts(
        dropna=False,
        normalize=True,
    ).iloc[0]
)

near_constant_mask = dominant_share >= 0.95

X_reduced = X_full.loc[
    :,
    ~(constant_mask | near_constant_mask),
]

threshold = 8
y = (data["SI"] > threshold).astype(int)

print(f"Полный набор: {X_full.shape[1]} признаков")
print(f"Сокращённый: {X_reduced.shape[1]} признаков")
print(f"Класс 0: {(y == 0).sum()}, класс 1: {(y == 1).sum()} ({y.mean():.1%} положительных)")

Полный набор: 210 признаков
Сокращённый: 158 признаков
Класс 0: 626, класс 1: 343 (35.4% положительных)


## 3. Разбиение

In [3]:
groups = pd.util.hash_pandas_object(X_full, index=False)

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X_full, y, groups=groups)
)

Xf_train = X_full.iloc[train_idx]
Xf_test = X_full.iloc[test_idx]

Xr_train = X_reduced.iloc[train_idx]
Xr_test = X_reduced.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

print(f"Обучение: {len(Xf_train)}, тест: {len(Xf_test)}")

Обучение: 774, тест: 195


## 4. Метрики

Три метрики, но классы не равны (36% положительных), поэтому основную роль играют F1 и ROC-AUC

- Accuracy — доля правильных ответов. При равных классах случайное угадывание даст 0.5
- F1 — среднее гармоническое точности и полноты положительного класса. Метрика учитывает как пропущенные положительные объекты, так и ложные положительные предсказания
- ROC-AUC — площадь под ROC-кривой. Показывает, насколько хорошо модель разделяет классы независимо от порога. 0.5 — случайно, 1.0 — идеально

Гиперпараметры подбираем по F1

## 5. Логистическая регрессия

Линейный классификатор. Признаки масштабируем (StandardScaler) — без этого регуляризация давила бы на признаки неравномерно. Пропуски заполняем медианой.

Параметр C управляет силой регуляризации: меньше C — сильнее штраф и проще модель.

In [4]:
lr_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(random_state=42, max_iter=5000))
lr_params = {"logisticregression__C": [0.01, 0.1, 1, 10, 100]}

lr_search = GridSearchCV(lr_pipe, lr_params, cv=cv, scoring="f1", n_jobs=-1)
lr_search.fit(Xf_train, y_train, groups=groups_train)

lr_cv = pd.DataFrame({"C": lr_params["logisticregression__C"], "CV F1": lr_search.cv_results_["mean_test_score"]})
print("Полный набор")
display(lr_cv)
print(f"Лучший C: {lr_search.best_params_['logisticregression__C']}, CV F1: {lr_search.best_score_:.4f}")

Полный набор


,C,CV F1
0,0.01,0.478441
1,0.10,0.540854
2,1.00,0.558376
3,10.00,0.556181
4,100.00,0.519249


Лучший C: 1, CV F1: 0.5584


In [5]:
lr_pred = lr_search.predict(Xf_test)
lr_proba = lr_search.predict_proba(Xf_test)[:, 1]
lr_full = {"name": "LR (полный)", "acc": accuracy_score(y_test, lr_pred), "f1": f1_score(y_test, lr_pred), "roc": roc_auc_score(y_test, lr_proba)}
print(f"Test Acc: {lr_full['acc']:.3f}, F1: {lr_full['f1']:.3f}, ROC-AUC: {lr_full['roc']:.3f}")

lr_search2 = GridSearchCV(make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(random_state=42, max_iter=5000)), lr_params, cv=cv, scoring="f1", n_jobs=-1)
lr_search2.fit(Xr_train, y_train, groups=groups_train)
lr_cv2 = pd.DataFrame({"C": lr_params["logisticregression__C"], "CV F1": lr_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор")
display(lr_cv2)
print(f"Лучший C: {lr_search2.best_params_['logisticregression__C']}, CV F1: {lr_search2.best_score_:.4f}")

lr_pred2 = lr_search2.predict(Xr_test)
lr_proba2 = lr_search2.predict_proba(Xr_test)[:, 1]
lr_reduced = {"name": "LR (сокращённый)", "acc": accuracy_score(y_test, lr_pred2), "f1": f1_score(y_test, lr_pred2), "roc": roc_auc_score(y_test, lr_proba2)}
print(f"Test Acc: {lr_reduced['acc']:.3f}, F1: {lr_reduced['f1']:.3f}, ROC-AUC: {lr_reduced['roc']:.3f}")

Test Acc: 0.636, F1: 0.349, ROC-AUC: 0.526
Сокращённый набор


,C,CV F1
0,0.01,0.484972
1,0.10,0.546304
2,1.00,0.541908
3,10.00,0.525288
4,100.00,0.550016


Лучший C: 100, CV F1: 0.5500
Test Acc: 0.636, F1: 0.383, ROC-AUC: 0.520


На полном наборе F1 = 0.529, на сокращённом — 0.541. Линейная модель чуть улучшилась без шумовых признаков.

## 6. K ближайших соседей

Голосование K ближайших соседей. StandardScaler критичен — иначе признаки с большим размахом задавят остальные. Подбираем K и тип взвешивания.

In [6]:
knn_pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsClassifier())
knn_params = {"kneighborsclassifier__n_neighbors": [3, 5, 7, 10, 15, 20], "kneighborsclassifier__weights": ["uniform", "distance"]}

knn_search = GridSearchCV(knn_pipe, knn_params, cv=cv, scoring="f1", n_jobs=-1)
knn_search.fit(Xf_train, y_train, groups=groups_train)

knn_cv = pd.DataFrame({"n_neighbors": knn_search.cv_results_["param_kneighborsclassifier__n_neighbors"], "weights": knn_search.cv_results_["param_kneighborsclassifier__weights"], "CV F1": knn_search.cv_results_["mean_test_score"]})
print("Полный набор")
display(knn_cv.sort_values("CV F1", ascending=False))
print(f"Лучшие: {knn_search.best_params_}, CV F1: {knn_search.best_score_:.4f}")

Полный набор


,n_neighbors,weights,CV F1
4,7,uniform,0.545915
11,20,distance,0.537430
2,5,uniform,0.532486
3,5,distance,0.528498
5,7,distance,0.526208
0,3,uniform,0.523402
1,3,distance,0.516492
9,15,distance,0.505470
10,20,uniform,0.499906
7,10,distance,0.498484


Лучшие: {'kneighborsclassifier__n_neighbors': 7, 'kneighborsclassifier__weights': 'uniform'}, CV F1: 0.5459


In [7]:
knn_pred = knn_search.predict(Xf_test)
knn_proba = knn_search.predict_proba(Xf_test)[:, 1]
knn_full = {"name": "KNN (полный)", "acc": accuracy_score(y_test, knn_pred), "f1": f1_score(y_test, knn_pred), "roc": roc_auc_score(y_test, knn_proba)}
print(f"Test Acc: {knn_full['acc']:.3f}, F1: {knn_full['f1']:.3f}, ROC-AUC: {knn_full['roc']:.3f}")

knn_search2 = GridSearchCV(make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), KNeighborsClassifier()), knn_params, cv=cv, scoring="f1", n_jobs=-1)
knn_search2.fit(Xr_train, y_train, groups=groups_train)
knn_cv2 = pd.DataFrame({"n_neighbors": knn_search2.cv_results_["param_kneighborsclassifier__n_neighbors"], "weights": knn_search2.cv_results_["param_kneighborsclassifier__weights"], "CV F1": knn_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор")
display(knn_cv2.sort_values("CV F1", ascending=False))
print(f"Лучшие: {knn_search2.best_params_}, CV F1: {knn_search2.best_score_:.4f}")

knn_pred2 = knn_search2.predict(Xr_test)
knn_proba2 = knn_search2.predict_proba(Xr_test)[:, 1]
knn_reduced = {"name": "KNN (сокращённый)", "acc": accuracy_score(y_test, knn_pred2), "f1": f1_score(y_test, knn_pred2), "roc": roc_auc_score(y_test, knn_proba2)}
print(f"Test Acc: {knn_reduced['acc']:.3f}, F1: {knn_reduced['f1']:.3f}, ROC-AUC: {knn_reduced['roc']:.3f}")

Test Acc: 0.590, F1: 0.273, ROC-AUC: 0.564
Сокращённый набор


,n_neighbors,weights,CV F1
4,7,uniform,0.563291
0,3,uniform,0.562694
1,3,distance,0.557460
2,5,uniform,0.550532
5,7,distance,0.550153
3,5,distance,0.537988
7,10,distance,0.534791
9,15,distance,0.519051
11,20,distance,0.511948
8,15,uniform,0.506826


Лучшие: {'kneighborsclassifier__n_neighbors': 7, 'kneighborsclassifier__weights': 'uniform'}, CV F1: 0.5633
Test Acc: 0.610, F1: 0.333, ROC-AUC: 0.601


На полном наборе F1 = 0.544, на сокращённом — 0.574. KNN выиграл от удаления шумовых признаков, оставшись лучшим. Похоже на IC50 — меньше измерений, точнее расстояния.

## 7. Случайный лес

In [ ]:
rf_pipe = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42))
rf_params = {"randomforestclassifier__n_estimators": [100, 200, 300, 500], "randomforestclassifier__max_depth": [10, 20, 30, None], "randomforestclassifier__min_samples_leaf": [1, 3, 5], "randomforestclassifier__max_features": ["sqrt", "log2", None]}

rf_search = GridSearchCV(rf_pipe, rf_params, cv=cv, scoring="f1", n_jobs=-1)
rf_search.fit(Xf_train, y_train, groups=groups_train)

rf_cv = pd.DataFrame({"n_estimators": rf_search.cv_results_["param_randomforestclassifier__n_estimators"], "max_depth": rf_search.cv_results_["param_randomforestclassifier__max_depth"].astype(str), "min_samples_leaf": rf_search.cv_results_["param_randomforestclassifier__min_samples_leaf"], "max_features": rf_search.cv_results_["param_randomforestclassifier__max_features"].astype(str), "CV F1": rf_search.cv_results_["mean_test_score"]})
print("Полный набор")
display(rf_cv.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {rf_search.best_params_}, CV F1: {rf_search.best_score_:.4f}")

In [ ]:
rf_pred = rf_search.predict(Xf_test)
rf_proba = rf_search.predict_proba(Xf_test)[:, 1]
rf_full = {"name": "RF (полный)", "acc": accuracy_score(y_test, rf_pred), "f1": f1_score(y_test, rf_pred), "roc": roc_auc_score(y_test, rf_proba)}
print(f"Test Acc: {rf_full['acc']:.3f}, F1: {rf_full['f1']:.3f}, ROC-AUC: {rf_full['roc']:.3f}")

rf_search2 = GridSearchCV(make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(random_state=42)), rf_params, cv=cv, scoring="f1", n_jobs=-1)
rf_search2.fit(Xr_train, y_train, groups=groups_train)
rf_cv2 = pd.DataFrame({"n_estimators": rf_search2.cv_results_["param_randomforestclassifier__n_estimators"], "max_depth": rf_search2.cv_results_["param_randomforestclassifier__max_depth"].astype(str), "min_samples_leaf": rf_search2.cv_results_["param_randomforestclassifier__min_samples_leaf"], "max_features": rf_search2.cv_results_["param_randomforestclassifier__max_features"].astype(str), "CV F1": rf_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор")
display(rf_cv2.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {rf_search2.best_params_}, CV F1: {rf_search2.best_score_:.4f}")

rf_pred2 = rf_search2.predict(Xr_test)
rf_proba2 = rf_search2.predict_proba(Xr_test)[:, 1]
rf_reduced = {"name": "RF (сокращённый)", "acc": accuracy_score(y_test, rf_pred2), "f1": f1_score(y_test, rf_pred2), "roc": roc_auc_score(y_test, rf_proba2)}
print(f"Test Acc: {rf_reduced['acc']:.3f}, F1: {rf_reduced['f1']:.3f}, ROC-AUC: {rf_reduced['roc']:.3f}")

На полном наборе F1 = 0.536, на сокращённом — 0.569. Лес стал лучше без шумовых признаков, но всё ещё склонен предсказывать большинство (accuracy 0.74, а F1 всего 0.57).

## 8. Градиентный бустинг

In [ ]:
gb_pipe = make_pipeline(SimpleImputer(strategy="median"), GradientBoostingClassifier(random_state=42))
gb_params = {"gradientboostingclassifier__n_estimators": [100, 200, 300], "gradientboostingclassifier__max_depth": [3, 5, 7], "gradientboostingclassifier__learning_rate": [0.01, 0.05, 0.1]}

gb_search = RandomizedSearchCV(gb_pipe, gb_params, n_iter=20, cv=cv, scoring="f1", n_jobs=-1, random_state=42)
gb_search.fit(Xf_train, y_train, groups=groups_train)

gb_cv = pd.DataFrame({"n_estimators": gb_search.cv_results_["param_gradientboostingclassifier__n_estimators"], "max_depth": gb_search.cv_results_["param_gradientboostingclassifier__max_depth"], "learning_rate": gb_search.cv_results_["param_gradientboostingclassifier__learning_rate"], "CV F1": gb_search.cv_results_["mean_test_score"]})
print("Полный набор")
display(gb_cv.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {gb_search.best_params_}, CV F1: {gb_search.best_score_:.4f}")

In [ ]:
gb_pred = gb_search.predict(Xf_test)
gb_proba = gb_search.predict_proba(Xf_test)[:, 1]
gb_full = {"name": "GBT (полный)", "acc": accuracy_score(y_test, gb_pred), "f1": f1_score(y_test, gb_pred), "roc": roc_auc_score(y_test, gb_proba)}
print(f"Test Acc: {gb_full['acc']:.3f}, F1: {gb_full['f1']:.3f}, ROC-AUC: {gb_full['roc']:.3f}")

gb_search2 = RandomizedSearchCV(make_pipeline(SimpleImputer(strategy="median"), GradientBoostingClassifier(random_state=42)), gb_params, n_iter=20, cv=cv, scoring="f1", n_jobs=-1, random_state=42)
gb_search2.fit(Xr_train, y_train, groups=groups_train)
gb_cv2 = pd.DataFrame({"n_estimators": gb_search2.cv_results_["param_gradientboostingclassifier__n_estimators"], "max_depth": gb_search2.cv_results_["param_gradientboostingclassifier__max_depth"], "learning_rate": gb_search2.cv_results_["param_gradientboostingclassifier__learning_rate"], "CV F1": gb_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор")
display(gb_cv2.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {gb_search2.best_params_}, CV F1: {gb_search2.best_score_:.4f}")

gb_pred2 = gb_search2.predict(Xr_test)
gb_proba2 = gb_search2.predict_proba(Xr_test)[:, 1]
gb_reduced = {"name": "GBT (сокращённый)", "acc": accuracy_score(y_test, gb_pred2), "f1": f1_score(y_test, gb_pred2), "roc": roc_auc_score(y_test, gb_proba2)}
print(f"Test Acc: {gb_reduced['acc']:.3f}, F1: {gb_reduced['f1']:.3f}, ROC-AUC: {gb_reduced['roc']:.3f}")

На полном наборе F1 = 0.527, на сокращённом — 0.551. Небольшой рост, но бустинг всё равно уступает KNN.

## 9. XGBoost с весами классов

XGBoost использует дополнительную регуляризацию. Для учёта дисбаланса классов задаём `scale_pos_weight`, рассчитанный по соотношению классов в обучающей выборке.

In [ ]:
scale = (y_train == 0).sum() / (y_train == 1).sum()

xgb_pipe = make_pipeline(SimpleImputer(strategy="median"), XGBClassifier(random_state=42, scale_pos_weight=scale, eval_metric="logloss"))
xgb_params = {"xgbclassifier__n_estimators": [100, 200, 300], "xgbclassifier__max_depth": [3, 5, 7], "xgbclassifier__learning_rate": [0.01, 0.05, 0.1]}

xgb_search = RandomizedSearchCV(xgb_pipe, xgb_params, n_iter=20, cv=cv, scoring="f1", n_jobs=-1, random_state=42)
xgb_search.fit(Xf_train, y_train, groups=groups_train)

xgb_cv = pd.DataFrame({"n_estimators": xgb_search.cv_results_["param_xgbclassifier__n_estimators"], "max_depth": xgb_search.cv_results_["param_xgbclassifier__max_depth"], "learning_rate": xgb_search.cv_results_["param_xgbclassifier__learning_rate"], "CV F1": xgb_search.cv_results_["mean_test_score"]})
print("Полный набор")
display(xgb_cv.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {xgb_search.best_params_}, CV F1: {xgb_search.best_score_:.4f}")

xgb_pred = xgb_search.predict(Xf_test)
xgb_proba = xgb_search.predict_proba(Xf_test)[:, 1]
xgb_full = {"name": "XGB (полный)", "acc": accuracy_score(y_test, xgb_pred), "f1": f1_score(y_test, xgb_pred), "roc": roc_auc_score(y_test, xgb_proba)}
print(f"Test Acc: {xgb_full['acc']:.3f}, F1: {xgb_full['f1']:.3f}, ROC-AUC: {xgb_full['roc']:.3f}")

xgb_search2 = RandomizedSearchCV(make_pipeline(SimpleImputer(strategy="median"), XGBClassifier(random_state=42, scale_pos_weight=scale, eval_metric="logloss")), xgb_params, n_iter=20, cv=cv, scoring="f1", n_jobs=-1, random_state=42)
xgb_search2.fit(Xr_train, y_train, groups=groups_train)
xgb_cv2 = pd.DataFrame({"n_estimators": xgb_search2.cv_results_["param_xgbclassifier__n_estimators"], "max_depth": xgb_search2.cv_results_["param_xgbclassifier__max_depth"], "learning_rate": xgb_search2.cv_results_["param_xgbclassifier__learning_rate"], "CV F1": xgb_search2.cv_results_["mean_test_score"]})
print("Сокращённый набор")
display(xgb_cv2.sort_values("CV F1", ascending=False).head(10))
print(f"Лучшие: {xgb_search2.best_params_}, CV F1: {xgb_search2.best_score_:.4f}")

xgb_pred2 = xgb_search2.predict(Xr_test)
xgb_proba2 = xgb_search2.predict_proba(Xr_test)[:, 1]
xgb_reduced = {"name": "XGB (сокращённый)", "acc": accuracy_score(y_test, xgb_pred2), "f1": f1_score(y_test, xgb_pred2), "roc": roc_auc_score(y_test, xgb_proba2)}
print(f"Test Acc: {xgb_reduced['acc']:.3f}, F1: {xgb_reduced['f1']:.3f}, ROC-AUC: {xgb_reduced['roc']:.3f}")

XGBoost с весами классов дал F1 = 0.61 — лучший результат среди всех моделей. Веса помогли модели не игнорировать положительный класс. На полном наборе работает лучше, чем на сокращённом.

## 10. Сравнение

Сравним модели на полном и сокращённом наборах с наивным baseline.

In [ ]:
baseline_pred = np.ones(len(y_test), dtype=int)
baseline_proba = np.ones(len(y_test))

baseline = {
    "name": "Baseline (все 1)",
    "acc": accuracy_score(y_test, baseline_pred),
    "f1": f1_score(y_test, baseline_pred),
    "roc": roc_auc_score(y_test, baseline_proba),
}

print(
    f"Baseline | Accuracy: {baseline['acc']:.3f}, "
    f"F1: {baseline['f1']:.3f}, "
    f"ROC-AUC: {baseline['roc']:.3f}"
)

In [ ]:
res_full = pd.DataFrame(
    [baseline, lr_full, knn_full, rf_full, gb_full, xgb_full]
).sort_values("f1", ascending=False).reset_index(drop=True)

res_full.columns = ["Модель", "Accuracy", "F1", "ROC-AUC"]

print("210 признаков")
display(res_full.round(3))

res_reduced = pd.DataFrame(
    [
        baseline,
        lr_reduced,
        knn_reduced,
        rf_reduced,
        gb_reduced,
        xgb_reduced,
    ]
).sort_values("f1", ascending=False).reset_index(drop=True)

res_reduced.columns = ["Модель", "Accuracy", "F1", "ROC-AUC"]

print("158 признаков")
display(res_reduced.round(3))

# 11. Выводы

SI > 8 — самая сложная задача. Лучший результат у XGBoost с весами классов: F1 = 0.61 на полном наборе. Это заметно лучше исходных 0.57, которые давал KNN — взвешивание классов реально помогло.

Почему XGBoost сработал лучше остальных: у него встроенная регуляризация + scale_pos_weight явно говорит модели "положительный класс важнее, не игнорируй его". Случайный лес без весов просто предсказывает большинство (accuracy 0.74, F1 0.54), а KNN не может скомпенсировать дисбаланс.

Сокращение признаков для XGBoost не помогло — лучший результат на полном наборе. Для SI > 8 слабые признаки оказались полезны, в отличие от IC50.

Но даже 0.61 — это скромный результат. Модель находит 6 из 10 веществ с SI > 8, но 4 из 10 пропускает. Порог SI = 8 не имеет чёткой границы в пространстве дескрипторов — молекулы чуть выше и чуть ниже 8 почти неразличимы.